In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("C:/Users/Gamer GTX/sintese/projeto trainee/Projeto-Trainee-I-Dados-2026.1/dados/data.csv")
LABELS_PATH = Path("C:/Users/Gamer GTX/sintese/projeto trainee/Projeto-Trainee-I-Dados-2026.1/dados/labels.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError("Defina o caminho para data.csv")

if not LABELS_PATH.exists():
    raise FileNotFoundError("Defina o caminho para labels.csv")

df_data = pd.read_csv(DATA_PATH)
df_labels = pd.read_csv(LABELS_PATH)

In [2]:
#concatenando os dois data frames
df_completo = pd.concat([df_data, df_labels], axis=1)

In [ ]:
#vendo se há linhas duplicadas
total_duplicatas = df_completo.duplicated().sum()
print(f'Total de linhas duplicadas: {total_duplicatas}')

Total de linhas duplicadas: 0


In [4]:
#descobrindo quantos dados nulos temos:
total_nulos = df_completo.isnull().sum().sum()
print(f'Total de dados nulos: {total_nulos}')

Total de dados nulos: 0


In [5]:
#estabalecendo uma variancia minima dos genes para reduzir possiveis ruidos no modelo
variancias = df_completo.var(numeric_only=True)
colunas_para_manter = variancias[variancias > 0.1].index.tolist()
colunas_para_manter.append('Class')
df_final = df_completo[colunas_para_manter]
print(f"Total de colunas ANTES do corte: {df_completo.shape[1]}")
print(f"Total de colunas DEPOIS do corte: {df_final.shape[1]}")

Total de colunas ANTES do corte: 20534
Total de colunas DEPOIS do corte: 19279


In [6]:
display(df_final)

,gene_1,gene_2,gene_3,gene_4,gene_6,gene_7,gene_10,gene_11,gene_12,gene_13,...,gene_20522,gene_20523,gene_20524,gene_20525,gene_20526,gene_20527,gene_20528,gene_20529,gene_20530,Class
0,2.017209,3.265527,5.478487,10.431999,7.175175,0.591871,0.591871,1.334282,2.015391,0.591871,...,8.210257,9.723516,7.220030,9.119813,12.003135,9.650743,8.921326,5.286759,0.000000,PRAD
1,0.592732,1.588421,7.586157,9.623011,6.816049,0.000000,0.000000,0.587845,2.466601,1.004394,...,7.323865,9.740931,6.256586,8.381612,12.674552,10.517059,9.397854,2.094168,0.000000,LUAD
2,3.511759,4.327199,6.881787,9.870730,6.972130,0.452595,0.000000,0.452595,1.981122,1.074163,...,8.127123,10.908640,5.401607,9.911597,9.045255,9.788359,10.090470,1.683023,0.000000,PRAD
3,3.663618,4.507649,6.659068,10.196184,7.843375,0.434882,0.000000,0.434882,2.874246,0.000000,...,8.792959,10.141520,8.942805,9.601208,11.392682,9.694814,9.684365,3.292001,0.000000,PRAD
4,2.655741,2.821547,6.539454,9.738265,6.566967,0.360982,0.000000,1.275841,2.141204,0.000000,...,8.891425,10.373790,7.181162,9.846910,11.922439,9.217749,9.461191,5.110372,0.000000,BRCA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
796,1.865642,2.718197,7.350099,10.006003,6.764792,0.496922,0.000000,0.000000,3.328722,0.000000,...,9.118313,10.004852,4.484415,9.614701,12.031267,9.813063,10.092770,8.819269,0.000000,BRCA
797,3.942955,4.453807,6.346597,10.056868,7.320331,0.000000,0.000000,1.049282,2.666211,0.000000,...,9.623335,9.823921,6.555327,9.064002,11.633422,10.317266,8.745983,9.659081,0.000000,LUAD
798,3.249582,3.707492,8.185901,9.504082,7.536589,1.811101,7.448149,4.049317,3.464198,0.586693,...,8.610704,10.485517,3.589763,9.350636,12.180944,10.681194,9.466711,4.677458,0.586693,COAD
799,2.590339,2.787976,7.318624,9.987136,9.213464,0.000000,1.578746,1.800703,3.635255,0.000000,...,8.605387,11.004677,4.745888,9.626383,11.198279,10.335513,10.400581,5.718751,0.000000,PRAD


In [ ]:
df_teste = df_final.select_dtypes(include=["number"])
#Análise dos quartis e dos outliers
outliers = 0
 
mediana = df_teste.median()
Q1 = df_teste.quantile(0.25)
Q3 = df_teste.quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - (1.5 * IQR)
limite_superior = Q3 + (1.5 * IQR)
outliers = (df_teste < limite_inferior) | (df_teste > limite_superior)
qtd_outliers = outliers.sum().sum()

display(f"Total de outliers: {qtd_outliers}")
    


'Total de outliers: 184656'

In [ ]:
#Utilizando os quartis para retirar os outliers e substituir os outliers pela mediana
colunas  = df_final.select_dtypes(include=["number"]).columns

medianas = df_final[colunas].median()
nao_outlier = (df_final[colunas] >= limite_inferior) & (df_final[colunas] <= limite_superior)
df_final[colunas] = df_final[colunas].where(nao_outlier, medianas, axis=1)

outliers = (df_final[colunas] < limite_inferior) | (df_final[colunas] > limite_superior)
qtd_outliers = outliers.sum().sum()
print(f"Total outliers: {qtd_outliers}")





Total outliers: 0


Para as inconsistências que podem levar a enviezamento, foram verificados os nulos e duplicados, não há nenhuma contradição pois as colunas contem apenas números, classes e o id do gene e não há vazamento de dados pois nenhuma variável é gerada como resultado.